# The ReAct Pattern: Reason + Act

ReAct (Yao et al. 2022) formalizes the interleaving of reasoning and action into a structured loop: Thought → Action → Observation → Thought → ... This pattern is the foundation for most production agent frameworks.

## Why Interleaving Matters

Two failure modes the ReAct pattern addresses:

**Pure action agents**: take actions without reasoning. Efficient but brittle — they cannot adapt when unexpected results occur, cannot explain their choices, and cannot detect when they are going off-course.

**Pure reasoning agents** (chain-of-thought only): reason extensively but cannot access external information or execute code. All knowledge is frozen in the model's parameters.

ReAct combines the strengths: the Thought step grounds the agent's reasoning in the current observation; the Action step grounds the observation in the real world. When a tool returns an unexpected result, the Thought step can diagnose and adapt.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional
import re, json

@dataclass
class ReActStep:
    step_num: int
    thought: str
    action_name: str
    action_args: dict
    observation: str

class ReActAgent:
    SYSTEM = (
        'You solve tasks by alternating Thought and Action.\n'
        'Format each step as:\n'
        'Thought: <reasoning>\n'
        'Action: <tool_name>[<args_json>]\n'
        'When done: Thought: I now know the answer.\nFinal Answer: <answer>'
    )

    def __init__(self, model_fn: Callable, tools: dict, max_steps: int = 10):
        self.model = model_fn
        self.tools = tools
        self.max_steps = max_steps

    def _parse(self, text: str) -> tuple:
        thought_m = re.search(r'Thought:\s*(.+?)(?=Action:|Final Answer:|$)', text, re.DOTALL)
        action_m = re.search(r'Action:\s*(\w+)\[(.+?)\]', text, re.DOTALL)
        final_m = re.search(r'Final Answer:\s*(.+)', text, re.DOTALL)
        thought = thought_m.group(1).strip() if thought_m else ''
        if final_m:
            return thought, 'final', {}, final_m.group(1).strip()
        if action_m:
            name = action_m.group(1)
            try:
                args = json.loads(action_m.group(2))
            except Exception:
                args = {'input': action_m.group(2).strip()}
            return thought, name, args, None
        return thought, None, {}, None

    def _call_tool(self, name: str, args: dict) -> str:
        if name not in self.tools:
            return f'Error: unknown tool {name}. Available: {list(self.tools)}'
        try:
            return str(self.tools[name](**args))
        except Exception as e:
            return f'Tool error: {e}'

    def run(self, task: str) -> dict:
        trajectory = []
        scratchpad = f'{self.SYSTEM}\n\nTask: {task}\n'
        for step in range(self.max_steps):
            response = self.model(scratchpad)
            thought, action, args, final = self._parse(response)
            if action == 'final' or final:
                return {'answer': final or action, 'steps': len(trajectory), 'trajectory': trajectory}
            if action:
                obs = self._call_tool(action, args)
                trajectory.append(ReActStep(step+1, thought, action, args, obs[:100]))
                scratchpad += f'Thought: {thought}\nAction: {action}[{json.dumps(args)}]\nObservation: {obs}\n'
            else:
                scratchpad += f'Thought: {thought}\n'
        return {'answer': 'max steps reached', 'steps': self.max_steps, 'trajectory': trajectory}

# Tools
def web_search(query: str) -> str:
    db = {
        'population paris': 'Paris has a population of approximately 2.1 million in the city proper.',
        'eiffel tower height': 'The Eiffel Tower is 330 meters tall including its antenna.',
        'france capital': 'The capital of France is Paris.',
    }
    for k, v in db.items():
        if any(w in query.lower() for w in k.split()):
            return v
    return 'No relevant results found.'

def calculator(expression: str) -> str:
    return str(eval(expression, {'__builtins__': {}}, {}))

tools = {'search': web_search, 'calculator': calculator}

step_n = [0]
def react_model(context):
    step_n[0] += 1
    if step_n[0] == 1:
        return 'Thought: I need to find the height of the Eiffel Tower.\nAction: search[{"query": "eiffel tower height"}]'
    if step_n[0] == 2:
        return 'Thought: The tower is 330m. I need to convert to feet: 330 * 3.28084.\nAction: calculator[{"expression": "330 * 3.28084"}]'
    return 'Thought: I now have the answer.\nFinal Answer: The Eiffel Tower is 330 meters (approximately 1083 feet) tall.'

agent = ReActAgent(react_model, tools)
result = agent.run('How tall is the Eiffel Tower in feet?')
print(f'Answer: {result["answer"]}')
print(f'Steps: {result["steps"]}')
for s in result['trajectory']:
    print(f'  Step {s.step_num}: [{s.action_name}] thought={s.thought[:40]}... obs={s.observation[:60]}...')

## ReAct vs Pure CoT vs Pure Tool Use

Empirical results from the original paper on HotpotQA, FEVER, AlfWorld:

- **CoT only**: strong on closed-book reasoning; fails when facts needed are not in training data; cannot update on new information
- **Act only**: efficient; fails when multi-hop reasoning is needed to combine tool results
- **ReAct**: best on knowledge-intensive QA and decision-making tasks; the thought steps explain failures, enabling more principled debugging

A key finding: the Thought step is not just for the model — it is the primary debugging tool for developers. When an agent fails, reading the thought trajectory usually reveals exactly where reasoning went wrong.

## Common ReAct Failure Modes

**Thought-action mismatch**: the thought says 'I will search for X' but the action calls a different tool or searches for Y. Caught by checking thought-action consistency.

**Observation ignoring**: the agent generates the next thought without incorporating the previous observation. Sign of poor context management.

**Hallucinated observations**: the model generates a plausible-sounding observation without actually calling the tool. Mitigation: separate tool call parsing from model generation; never let the model generate observations.

**Infinite loops**: agent calls the same tool with the same arguments repeatedly. Fix: detect repeated (action, args) pairs and break out.